# Two sequential cost-adapter curricula

Select a **GPU runtime**, then choose **Runtime → Run all**. This runs two independent curricula: the original temporal-pair adapter and the oracle-clean-pair adapter. Each policy collects its own new experience and continues from its previous weights.

Starts keep **40 ≤ x ≤ 54**. New y bands are **[22,30] → [14,22] → [8,14]**, below the original near band [30,36]. Goal (50,42), frozen encoder/world model and planner settings are inherited. Original runs remain read-only. Outputs go to `multiband_curriculum_v1`.

This is a larger experiment: up to **840 episodes**, including raw references, both curricula's collection, pre-update tests and retention tests. Stages print elapsed times and resume completed cases. Actual results may improve or worsen; the loop measures both.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, shutil, signal, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
DRIVE = Path('/content/drive/MyDrive')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_REF = 'curriculum-cost-adapter'
SOURCE_COMMIT = 'a854e170c114d4977b91171891201111be16a221'
REPO = Path('/content/i-jepa-wm-curriculum')
DINO_ROOT = Path('/content/dino_wm')
PARENT = DRIVE / 'datasets/curriculum_cost_adapter/pilot_v1'
CLEAN = DRIVE / 'datasets/curriculum_cost_adapter/clean_pair_v1'
ADJACENT = DRIVE / 'datasets/curriculum_cost_adapter/adjacent_band_v1'
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/multiband_curriculum_v1'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE = DRIVE / 'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS = Path('/content/curriculum-multiband-artifacts')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'

def shell(command, *, cwd=None, log=None):
    command = [str(arg) for arg in command]
    print('$', ' '.join(command), flush=True)
    stream = open(log, 'a', buffering=1) if log else None
    try:
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    if stream: stream.write(line)
                returncode = process.wait()
            except BaseException:
                process.send_signal(signal.SIGINT)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: process.terminate()
                raise
        if returncode: raise subprocess.CalledProcessError(returncode, command)
    finally:
        if stream: stream.close()


## Source and frozen-model setup

The notebook pins the evaluation/training source revision, validates all three previous experiments and records checkpoint lineage. It uses the same saved goal tokens for raw scoring, both curricula and every heatmap. Keep the output directory and pinned source revision for resumption.


In [ ]:
for source in (PARENT, CLEAN, ADJACENT):
    assert source.is_dir(), f'Missing completed experiment: {source}'
    assert OUTPUT.resolve() != source.resolve() and not OUTPUT.resolve().is_relative_to(source.resolve()), 'Output must be outside both sources'
assert (CLEAN / 'run_0/goal_tokens.pt').is_file(), 'Run the clean-pair notebook first'
if not REPO.exists():
    shell(['git', 'clone', '--branch', SOURCE_REF, '--single-branch', REPO_URL, REPO])
else:
    assert (REPO / '.git').exists(), f'{REPO} is not a Git checkout'
    assert subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip() == REPO_URL
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip(), 'Source clone has local changes'
    shell(['git', '-C', REPO, 'fetch', 'origin', f'refs/heads/{SOURCE_REF}:refs/remotes/origin/{SOURCE_REF}'])
shell(['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT or f'origin/{SOURCE_REF}'])
source_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Multi-band source revision:', source_commit)

shell([sys.executable, '-m', 'pip', 'install', '-q', 'gym==0.26.2', 'omegaconf', 'pyyaml', 'scipy', 'einops', 'matplotlib', 'imageio', 'imageio-ffmpeg', 'pillow'])
if not DINO_ROOT.exists(): shell(['git', 'clone', 'https://github.com/gaoyuezhou/dino_wm.git', DINO_ROOT])
assert not subprocess.check_output(['git', '-C', str(DINO_ROOT), 'status', '--porcelain', '--untracked-files=no'], text=True).strip(), 'DINO-WM source has local changes'
shell(['git', '-C', DINO_ROOT, 'checkout', DINO_COMMIT])
ARTIFACTS.mkdir(parents=True, exist_ok=True)
ENCODER, CHECKPOINT = ARTIFACTS / 'encoder.pth.tar', ARTIFACTS / 'fixed-best.pt'
for source, destination in ((ENCODER_DRIVE, ENCODER), (WM_DRIVE, CHECKPOINT)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not destination.exists() or destination.stat().st_size != source.stat().st_size: shutil.copy2(source, destination)
COMMON = ['--parent-dir', str(PARENT), '--clean-dir', str(CLEAN), '--adjacent-dir', str(ADJACENT), '--output-dir', str(OUTPUT),
          '--world-model-checkpoint', str(CHECKPOINT), '--encoder-checkpoint', str(ENCODER),
          '--dino-wm-root', str(DINO_ROOT), '--device', 'cuda', '--precision', 'fp32', '--encoder-batch-size', '8']
def stage(name, recipe=None, band_index=None):
    extra = ['--recipe', recipe] if recipe else []
    if band_index is not None: extra += ['--band-index', str(band_index)]
    started = time.monotonic()
    shell([sys.executable, '-m', 'src.mylonas_ijepa_wm.curriculum_multiband', name, *COMMON, *extra],
          cwd=REPO)
    print(f"{name} {recipe or ''} {band_index or ''} took {(time.monotonic() - started) / 60:.1f} minutes")
print('GPU:', torch.cuda.get_device_name())
print('Torch:', torch.__version__)
print('Parent:', PARENT)
print('Clean adapter source:', CLEAN)
print('Adjacent evaluation source:', ADJACENT)
print('New output:', OUTPUT)


## Prepare fixed cases, cache the heatmap grid, and measure raw cost

The 20 original near cases and 20 adjacent cases are reused exactly. Lower-band cases are generated once, as are 40 separate collection cases per band, at least 0.25 units from every evaluation start. Both policies receive the same collection starts and seeds, but produce separate trajectories.

The heatmap grid contains 176 real rendered positions: x=40,42,…,54; y=8,10,…,50. Encoding is cached once. Grid tokens are diagnostic data and never enter replay or training. Rendering probes execute zero primitive actions. The raw baseline evaluates all four bands once (80 episodes).


In [ ]:
stage('prepare')
manifest = json.loads((OUTPUT / 'manifest.json').read_text())
print('Bands:', manifest['bounds'])
print('Evaluation cases:', {key: len(rows) for key, rows in manifest['evaluation'].items()})
print('Collection attempts per curriculum:', {key: len(rows) for key, rows in manifest['collection'].items()})
stage('grid')
stage('baseline')


## Collect experience and update each curriculum

For every band: evaluate the incoming adapter, collect 40 attempts with it frozen, encode successful experience, continue fitting for 500 updates, and evaluate the new checkpoint on the current and earlier bands. Replay samples a band uniformly, then an episode and a pair. Earlier successes remain available. Each fit uses fresh Adam moments and the existing adapter weights.

The original recipe keeps its ordinary temporal pairs. The clean recipe additionally requires ≥0.5 units of true goal progress. Failure to find usable new pairs stops that curriculum; the other still runs. Evaluation losses are reported and do not automatically select checkpoints or change the schedule.

After every band update, three maps show raw MSE, adapted MSE, and their difference. All versions use the same grid, goal and display scale. Colorbar extensions and printed saturation counts expose values outside that scale; JSON files retain the actual values. White dots are training replay positions, the star is the goal, and the orange outline is the most recently learned band. These are real-image cost maps, not world-model rollout predictions.


In [ ]:
from IPython.display import display, Image

def show_heatmap(recipe, version):
    folder = OUTPUT / f'branches/{recipe}/heatmaps'
    data = json.loads((folder / f'v{version}.json').read_text())
    print(f"\n{recipe.upper()} v{version}: fixed display max={data['display_max']:.5g}; "
          f"actual adapted max={data['adapted_max']:.5g}; "
          f"points above scale={data['saturated_adapted_points']}/176")
    print('Difference range:', data['difference_min'], data['difference_max'], 'points beyond difference scale:', data['saturated_difference_points'])
    display(Image(filename=str(folder / f'v{version}.png')))

errors = {}
for recipe in ('original', 'clean'):
    print('\n' + '=' * 70 + '\nCURRICULUM:', recipe)
    try:
        stage('initial', recipe)
        show_heatmap(recipe, 0)
        for band_index in (1, 2, 3):
            print(f'\n{recipe}: learning in band {band_index}')
            stage('band', recipe, band_index)
            stopped = OUTPUT / f'branches/{recipe}/stopped.json'
            if stopped.exists():
                print('THIS CURRICULUM STOPPED:', json.dumps(json.loads(stopped.read_text()), indent=2))
                break
            folder = OUTPUT / f'branches/{recipe}/updates/v{band_index}'
            status = json.loads((folder / 'status.json').read_text())
            training = json.loads((folder / 'training.json').read_text())
            print(f"Collection successes: {status['collection_successes']}/{status['collection_attempts']}")
            print(f"Replay episodes: {status['old_episodes']} earlier + {status['new_episodes']} new")
            print('Episodes per band:', training['band_episode_counts'])
            print('Actual sampled pairs per band:', training['band_sample_counts'])
            print('Ranking loss, first/last:', training['history'][0]['rank_loss'], training['history'][-1]['rank_loss'])
            print('Incoming checkpoint:', status['inputs']['incoming_sha256'])
            print('Updated checkpoint:', status['checkpoint_sha256'])
            show_heatmap(recipe, band_index)
            display(Image(filename=str(folder / 'training_and_experience.png')))
    except subprocess.CalledProcessError as error:
        errors[recipe] = str(error)
        print(f'{recipe} encountered an execution error. Its saved artifacts are preserved; continuing with the other curriculum.')
print('\nExecution errors:', errors or 'none')


## Compare transfer, within-band learning and retention

Rows are adapter versions; columns are evaluation bands. A version's next-band test measures performance before collecting experience there. Tests after an update show learning in the current band and any forgetting in earlier bands. N/A marks missing or not-yet-run evaluations.

Paired wins/losses compare identical starts and CEM seeds. The inherited ≥18/20 reference competence and ≤2 paired losses are operational annotations, not significance tests or progression gates. At a 20/20 ceiling, examine actions, final distance and paths too. The monitoring cases repeat across versions; they are not independent new tasks.


In [ ]:
stage('report')
report = json.loads((OUTPUT / 'report.json').read_text())
bands = ('near', 'band_1', 'band_2', 'band_3')
print('\nRAW REFERENCE')
if report.get('raw_error'): print('Raw results unavailable:', report['raw_error'])
for band in bands:
    row = report['raw'][band]
    print(band, row)
for recipe in ('original', 'clean'):
    branch = report['branches'][recipe]
    print(f"\n{recipe.upper()} — status: {branch['status']}")
    if branch.get('error'): print('Results unavailable:', branch['error'])
    print('Each cell below: successes / primitive actions; N/A = no completed evaluation')
    print('version      near         band 1       band 2       band 3')
    for version, rows in branch['versions'].items():
        cells = ['N/A' if rows[band] is None else f"{rows[band]['successes']}/20 | {rows[band]['primitive_steps']}" for band in bands]
        print(f"v{version:<7}" + ''.join(f'{cell:<14}' for cell in cells))
        print('  Mean final distances:', {band: None if rows[band] is None else round(rows[band]['mean_final_distance'], 3) for band in bands})
    print('Collection:', json.dumps(branch['collections'], indent=2))
    print('Paired changes vs raw and incoming checkpoint:', json.dumps(branch['comparisons'], indent=2))
    print('Primitive-action journal:', json.dumps(branch['budget'], indent=2))
    if branch['stopped']: print('Stopped reason:', branch['stopped'])
print('\nTotal primitive interactions:', report['total_primitive_interactions'])
print('Known primitive interactions:', report['known_primitive_interactions'], '| total is a lower bound/incomplete:', report['total_count_is_lower_bound'])
print('Heatmap rendering primitive interactions:', report['diagnostic_primitive_interactions'])
print('Interpretation:', report['note'])
display(Image(filename=str(OUTPUT / 'curriculum_matrix.png')))
print('Full report:', OUTPUT / 'report.json')


## Return the executed result

Download this executed file using **File → Download → Download .ipynb** and send it back. Checkpoints, raw episodes, replay, fixed cases, losses, numeric heatmaps, figures and journals persist under `MyDrive/datasets/curriculum_cost_adapter/multiband_curriculum_v1`.

If a runtime disconnects, rerun this notebook with the same pinned source and output path. Completed cases and committed updates are validated and reused. Incomplete cases are retried with their previous actions still counted; an interrupted fit restarts its fixed schedule from the incoming adapter. A curriculum stopped for insufficient new experience remains stopped for interpretation.
